In [10]:
# kaufman_efficiency_ratio ER
import pandas as pd
import numpy as np

def kaufman_efficiency_ratio(close_series, period=10):
    # Variation nette sur la période (entre maintenant et il y a N périodes)
    net_change = np.abs(close_series - close_series.shift(period))
    
    # Somme des variations absolues bar à bar sur la période
    change = np.abs(close_series - close_series.shift(1))
    volatility = change.rolling(window=period).sum()
    
    # Éviter la division par zéro
    er = np.where(volatility == 0, 0, net_change / volatility)
    
    return pd.Series(er, index=close_series.index)

# Exemple d'utilisation :
# df['KER'] = kaufman_efficiency_ratio(df['Close'], period=10)

In [1]:
# swing structure
import numpy as np
import pandas as pd


def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = "NEUTRAL"

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                last_confirmed_high = None
                last_confirmed_low = None
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [54]:
# rsi ranging stratey
from backtesting import Strategy

global_tp_multiple_rsi = 0
global_stop_trade_hour = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple_rsi = 5
    start_trade_hour = 5
    stop_trade_hour = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False

        if global_tp_multiple_rsi != 0:
            self.tp_multiple_rsi = global_tp_multiple_rsi

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        planned_entry_price = self.data.High[-1]
        sl_price = self.last_low
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.buy(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        planned_entry_price = self.data.Low[-1]
        sl_price = self.last_high
        per_share_risk = abs(planned_entry_price - sl_price)
        tp_multiple_rsi = self.tp_multiple_rsi 
        tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
        position_size = int((self.risk_percent * self.equity) / per_share_risk)

        # print(candle_date)
        # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
        if per_share_risk < self.data['ATR'][-1] * 2.5 and per_share_risk > self.data['ATR'][-1] * 1:
            # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
            self.order_placed_hour = candle_date.time().hour
            self.sell(size = position_size, stop=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
            self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        c_long = self.data['RSI14'][-1] < 35
        c_short = self.data['RSI14'][-1] > 65

        if not self.current_day_order_placed:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if c_long:
                self.place_long_order(candle_datetime)

            # if c_short:
            #     self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()
                        self.current_day_order_placed = False
                        if order.is_long:
                            self.place_long_order(candle_datetime)
                        else:
                            self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_hour:
            self.reset_range(current_bar_date)

        if self.last_low is None:
            self.last_low = self.data.Low[-1]
        else:
            if self.last_low > self.data.Low[-1]:
                self.last_low = self.data.Low[-1]

        if self.last_high is None:
            self.last_high = self.data.High[-1]
        else:
            if self.last_high < self.data.High[-1]:
                self.last_high = self.data.High[-1]

        c_market_structure = self.data['market_state'][-1] == 'NEUTRAL'
        c_er = self.data['ER'][-1] < self.er15_threshold
        c_time_limit = candle_datetime.time().hour < self.stop_trade_hour and candle_datetime.time().hour > self.start_trade_hour
        
        if c_er and c_market_structure and c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [59]:
from datetime import timedelta

from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator

window = 3

global_tp_multiple_rsi = 1
start_date = '2020-11-1 00:00'
end_date = '2023-2-28 00:00'
# start_date = '2020-11-1 00:00'
# end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

# df_w1 = pd.read_csv(f"./Trading Data/{pair}_W1.csv",sep='\t', parse_dates=['Date'], index_col='Date')
# df_w1 = df_w1.loc[start_date : end_date]

# df_result_w1 = implement_market_structure_states_strict(df_w1, n=window)
#Affichage des transitions d'états
# df_result_w1["state_changed"] = df_result_w1["market_state"] != df_result_w1[
#     "market_state"
# ].shift(1)

# for index in range(len(df_result_w1[df_result_w1["state_changed"]])):
#     print(df_result_w1[df_result_w1["state_changed"]].iloc[index])

df_d1 = pd.read_csv(f"./Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

# df_w1_direction_list = []
# for i in range(len(df_result_w1)):
#     row = df_result_w1.iloc[i]
#     start_week = row.name
#     end_week = start_week + timedelta(days=6)
#     df_by_day = df_d1.loc[f'{start_week}' : f'{end_week}']
#     for index2, row2 in df_by_day.iterrows():
#         df_w1_direction_list.append(df_result_w1.iloc[i-1]['market_state'])

indicator_er = kaufman_efficiency_ratio(df_d1['Close'], 20)
df_result_d1 = implement_market_structure_states_strict(df_d1, n=window)

df_result_d1['ER'] = indicator_er.values

df_h1 = pd.read_csv(f"./Trading Data/{pair}_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

h1_tf_direction_list = []
h1_tf_direction_w1_list = []
h1_tf_er15_list = []

for i in range(len(df_result_d1)):
    row = df_result_d1.iloc[i]

    df_by_day = df_h1.loc[f'{row.name.date()}' : f'{row.name.date()}']
    for index2, row2 in df_by_day.iterrows():
        h1_tf_direction_list.append(df_result_d1.iloc[i-1]['market_state'])
        # h1_tf_direction_w1_list.append(df_result_d1.iloc[i-1]['market_state_w1'])
        h1_tf_er15_list.append(df_result_d1.iloc[i-1]['ER'])

# print(len(h1_tf_er15_list))
df_h1['market_state'] = h1_tf_direction_list
# df_h1['market_state_w1'] = h1_tf_direction_w1_list
df_h1['ER'] = h1_tf_er15_list
indicator_atr_14 = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_h1['Close'], window=14)
df_h1['RSI14'] = indicator_rsi_14._rsi
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2022-01-24 14:00:00+00:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2022-08-11 09:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

Start                     2020-11-01 22:00...
End                       2023-02-28 00:00...
Duration                    848 days 02:00:00
Exposure Time [%]                     1.50951
Equity Final [$]                     9333.962
Equity Peak [$]                     10355.993
Return [%]                           -6.66038
Buy & Hold Return [%]                21.38914
Return (Ann.) [%]                    -2.36085
Volatility (Ann.) [%]                  3.6401
CAGR [%]                             -2.92484
Sharpe Ratio                         -0.64857
Sortino Ratio                        -0.79602
Calmar Ratio                         -0.19967
Alpha [%]                            -7.54695
Beta                                  0.04145
Max. Drawdown [%]                   -11.82352
Avg. Drawdown [%]                     -1.7681
Max. Drawdown Duration      537 days 06:00:00
Avg. Drawdown Duration       80 days 00:00:00
# Trades                                   38
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '4h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p6089', ...)